In [24]:
import kagglehub
path=kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")

Using Colab cache for faster access to the 'fruits-fresh-and-rotten-for-classification' dataset.


In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
import os
train_path = os.path.join(path, 'dataset','train')

In [ ]:
img_height = 224
img_width = 224
batch_size = 32

In [ ]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

In [ ]:
original_class_name_for_display = train_ds.class_names

In [ ]:
original_class_name_for_display

In [ ]:
def normalize(image,lable):
  image = tf.cast(image/255. , tf.float32)
  return image,lable

In [ ]:
train_ds =  train_ds.map(normalize)
val_ds =  val_ds.map(normalize)

In [ ]:
print("Dataset loaded and normalized.")
print(f"Number of training batches: {len(train_ds)}")
print(f"Number of validation batches: {len(val_ds)}")
print(f"Image shape: ({img_height}, {img_width}, 3)") # Assuming 3 color channels
print(f"Batch size: {batch_size}")
print("Original classes found:", original_class_name_for_display)

In [ ]:
def map_binary(image,lable):
  binary_lable = tf.cast(lable>=len(original_class_name_for_display)//2,tf.int32)
  return image,binary_lable

In [ ]:
train_ds = train_ds.map(map_binary)
val_ds = val_ds.map(map_binary)

In [ ]:
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip('horizontal'),
  tf.keras.layers.RandomRotation(0.2),
  tf.keras.layers.RandomZoom(0.2),
  tf.keras.layers.RandomContrast(0.2)
])

def augment(image,lable):
  image = data_augmentation(image)
  return image,lable

train_ds_augmented = train_ds.map(augment)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32,(3,3),activation='relu',input_shape=(img_height,img_width,3)),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(64,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Conv2D(128,(3,3),activation='relu'),
    tf.keras.layers.MaxPooling2D(2,2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [18]:
--
  +.epochs = 30
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True)
history = model.fit(
    train_ds_augmented,
    validation_data=val_ds,
    epochs=epochs,
    callbacks=[early_stopping]
)

Epoch 1/30
273/273 ━━━━━━━━━━━━━━━━━━━━ 1364s 5s/step - accuracy: 0.8531 - loss: 0.3246 - val_accuracy: 0.9183 - val_loss: 0.2002
Epoch 2/30
273/273 ━━━━━━━━━━━━━━━━━━━━ 1402s 5s/step - accuracy: 0.9137 - loss: 0.2094 - val_accuracy: 0.9468 - val_loss: 0.1421
Epoch 3/30
273/273 ━━━━━━━━━━━━━━━━━━━━ 1359s 5s/step - accuracy: 0.9331 - loss: 0.1685 - val_accuracy: 0.9367 - val_loss: 0.1639
Epoch 4/30
273/273 ━━━━━━━━━━━━━━━━━━━━ 1353s 5s/step - accuracy: 0.9326 - loss: 0.1642 - val_accuracy: 0.9431 - val_loss: 0.1321
Epoch 5/30
273/273 ━━━━━━━━━━━━━━━━━━━━ 1383s 5s/step - accuracy: 0.9389 - loss: 0.1486 - val_accuracy: 0.9560 - val_loss: 0.1080
Epoch 6/30
273/273 ━━━━━━━━━━━━━━━━━━━━ 1327s 5s/step - accuracy: 0.9485 - loss: 0.1250 - val_accuracy: 0.9518 - val_loss: 0.1078
Epoch 7/30
273/273 ━━━━━━━━━━━━━━━━━━━━ 1326s 5s/step - accuracy: 0.9489 - loss: 0.1273 - val_accuracy: 0.9307 - val_loss: 0.1791
Epoch 8/30
273/273 ━━━━━━━━━━━━━━━━━━━━ 1329s 5s/step - accuracy: 0.9522 - loss: 0.1185 - 

In [19]:
test_path = os.path.join(path, 'dataset','test')
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    shuffle=False
)

Found 2698 files belonging to 6 classes.


In [20]:
test_ds = test_ds.map(normalize)
test_ds = test_ds.map(map_binary)

In [21]:
loss , accuracy = model.evaluate(test_ds)
test_loss = loss
test_accuracy =accuracy
test_loss

85/85 ━━━━━━━━━━━━━━━━━━━━ 99s 1s/step - accuracy: 0.9633 - loss: 0.0917


0.09170462936162949

In [22]:
model_save_path = 'fruits_classification.h5'
model.save(model_save_path)

In [23]:
model_save_path = 'fruits_classification.keras'
model.save(model_save_path)